# Data Challenge HUN 2026 — Modelo 2: duración documentada de episodios

## Sección A — Introducción

Este notebook investiga si `documented_duration_min` puede utilizarse como variable candidata para estudiar y, eventualmente, pronosticar la duración de un episodio reconstruido en urgencias. El análisis se concentra exclusivamente en `episode_gap_threshold_h = 24`; el criterio de 48 horas se utiliza únicamente como sensibilidad y nunca se combina con el conjunto principal.

Los episodios son inferidos a partir de registros OMOP. La duración documentada describe el intervalo reconstruido entre `episode_start` y `episode_end`, pero no ha sido validada como tiempo real de estancia, espera hasta primera atención, duración de atención médica ni permanencia total en urgencias. Cualquier utilidad operacional requiere confirmación con el hospital.

La prioridad es establecer primero la calidad y el significado de la variable. Solo si existe evidencia externa suficiente se debería habilitar un modelo predictivo; la existencia de datos y la posibilidad técnica de entrenar un algoritmo no constituyen evidencia de utilidad clínica.

## Sección B — Herramientas y conexión segura

Se emplean pandas, NumPy, matplotlib, scikit-learn, SQLAlchemy, psycopg y joblib cuando son pertinentes. La fuente prevista es PostgreSQL local (`hun_analitica`, esquema `urgencias`); la contraseña se solicita de forma interactiva mediante `getpass` y nunca se imprime ni se almacena.

Para permitir una ejecución local reproducible cuando no se dispone de la contraseña, el notebook admite explícitamente `HUN_MODELO2_FUENTE=csv_local`. Este respaldo usa el archivo local `urgencias_episodios.csv`, que contiene las mismas columnas declaradas. La modalidad utilizada queda impresa y se reporta, porque un diagnóstico sobre CSV no equivale a una extracción ejecutada desde PostgreSQL.

In [ ]:
import os
import getpass
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

BASE = Path.cwd()
if not (BASE / "urgencias_episodios.csv").exists():
    BASE = Path("..").resolve()

RESULTADOS = BASE / "resultados" / "duracion"
FIGURAS = RESULTADOS / "figuras"
RESULTADOS.mkdir(parents=True, exist_ok=True)
FIGURAS.mkdir(parents=True, exist_ok=True)

UMBRAL_PRINCIPAL = 24
UMBRAL_SENSIBILIDAD = 48
FUENTE = os.getenv("HUN_MODELO2_FUENTE", "postgresql").lower()
permitir_modelado = False

print("Fuente seleccionada:", FUENTE)
print("Umbral principal:", UMBRAL_PRINCIPAL, "horas")
print("Modelado habilitado:", permitir_modelado)

### Carga controlada y comprobación de esquema

La consulta principal selecciona únicamente campos necesarios para calidad, descripción y definición de predictores candidatos. No selecciona `visit_occurrence_id` ni `person_id`, y no imprime identificadores. La comprobación de columnas se hace contra `information_schema` y es de solo lectura.

El filtro de 24 horas se aplica antes del análisis principal. Las columnas de detalle y los indicadores finales se conservan para diagnóstico retrospectivo, pero se excluyen explícitamente de cualquier predictor disponible al comienzo del episodio.

In [ ]:
COLUMNAS_NECESARIAS = [
    "episode_id", "episode_number", "episode_start", "episode_end",
    "documented_duration_min", "detail_count", "positive_detail_count",
    "zero_detail_count", "has_emergency_detail", "has_diagnostic_detail",
    "has_inpatient_detail", "has_rehabilitation_detail", "has_positive_duration",
    "is_zero_duration", "is_over_24h", "is_over_48h", "episode_gap_threshold_h",
]

def cargar_postgresql():
    password = getpass.getpass("Contraseña de PostgreSQL: ")
    url = URL.create(
        drivername="postgresql+psycopg",
        username="postgres",
        password=password,
        host="localhost",
        port=5432,
        database="hun_analitica",
    )
    engine = create_engine(url)
    with engine.connect() as conexion:
        tabla = conexion.execute(text("""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_schema = 'urgencias' AND table_name = 'episodios'
            ORDER BY ordinal_position
        """)).fetchall()
        columnas = {fila[0] for fila in tabla}
        faltantes = sorted(set(COLUMNAS_NECESARIAS) - columnas)
        if faltantes:
            raise ValueError(f"Faltan columnas requeridas: {faltantes}")
    consulta = f"""
        SELECT {", ".join(COLUMNAS_NECESARIAS)}
        FROM urgencias.episodios
        WHERE episode_gap_threshold_h = {UMBRAL_PRINCIPAL}
        ORDER BY episode_start
    """
    datos = pd.read_sql(consulta, engine)
    del password
    return datos, "postgresql"


def cargar_csv_local():
    archivo = BASE / "urgencias_episodios.csv"
    if not archivo.exists():
        raise FileNotFoundError(f"No existe el respaldo local: {archivo}")
    datos = pd.read_csv(archivo, usecols=COLUMNAS_NECESARIAS, low_memory=False)
    datos = datos[datos["episode_gap_threshold_h"] == UMBRAL_PRINCIPAL].copy()
    return datos, "csv_local"


if FUENTE == "csv_local":
    df = cargar_csv_local()[0]
    fuente_ejecutada = "csv_local"
else:
    df, fuente_ejecutada = cargar_postgresql()

df["episode_start"] = pd.to_datetime(df["episode_start"], errors="coerce")
df["episode_end"] = pd.to_datetime(df["episode_end"], errors="coerce")
df["documented_duration_min"] = pd.to_numeric(df["documented_duration_min"], errors="coerce")
print("Fuente ejecutada:", fuente_ejecutada)
print("Episodios con umbral 24:", len(df))

## Sección C — Calidad de datos

Esta etapa cuantifica completitud, coherencia temporal, ceros, valores negativos, extremos y duplicados. La duración se contrasta con la diferencia entre inicio y fin usando una tolerancia numérica pequeña, porque la fuente puede contener representación decimal de minutos. Ningún valor extremo se elimina automáticamente.

Los conteos se calculan sin mostrar filas individuales ni identificadores. Los indicadores `detail_count` y los demás campos de detalle se interpretan como evidencia retrospectiva acumulada, no como información disponible necesariamente al inicio.

In [ ]:
duracion = df["documented_duration_min"]
diferencia_min = (df["episode_end"] - df["episode_start"]).dt.total_seconds() / 60
coherente = (diferencia_min - duracion).abs() <= 1e-6

calidad = pd.Series({
    "total_episodios": len(df),
    "fecha_minima": df["episode_start"].min(),
    "fecha_maxima": df["episode_start"].max(),
    "nulos_totales": int(df.isna().sum().sum()),
    "duraciones_negativas": int((duracion < 0).sum()),
    "duraciones_cero": int((duracion == 0).sum()),
    "duraciones_positivas": int((duracion > 0).sum()),
    "episodios_mas_24h": int((duracion > 24 * 60).sum()),
    "episodios_mas_48h": int((duracion > 48 * 60).sum()),
    "episodios_mas_7_dias": int((duracion > 7 * 24 * 60).sum()),
    "episode_id_duplicados": int(df["episode_id"].duplicated().sum()),
    "episode_end_anterior_a_start": int((df["episode_end"] < df["episode_start"]).sum()),
    "duracion_coherente_con_fechas": int(coherente.sum()),
    "duracion_incoherente_con_fechas": int((~coherente & duracion.notna() & diferencia_min.notna()).sum()),
    "proporcion_duracion_positiva": float((duracion > 0).mean()),
}, dtype=object)

print(calidad.to_string())
calidad.to_frame("valor").to_csv(RESULTADOS / "resumen_calidad.csv", index_label="indicador")

## Sección D — Análisis descriptivo de duraciones

Se resumen todos los episodios y, por separado, los episodios con duración positiva. La separación no modifica el conjunto original: permite mostrar cuánto influyen los ceros en la media y la mediana. La transformación `log1p` se usa únicamente para visualizar la asimetría y no cambia `documented_duration_min`.

Los histogramas y el boxplot se guardan localmente. La serie mensual utiliza mediana y percentil 90, dos medidas menos sensibles que la media a valores extremadamente largos, aunque no eliminan la necesidad de investigar esos extremos.

In [ ]:
def estadisticas_serie(serie):
    return pd.Series({
        "n": int(serie.notna().sum()),
        "media_min": serie.mean(),
        "mediana_min": serie.median(),
        "desviacion_estandar_min": serie.std(),
        "p25_min": serie.quantile(0.25),
        "p75_min": serie.quantile(0.75),
        "p90_min": serie.quantile(0.90),
        "p95_min": serie.quantile(0.95),
        "p99_min": serie.quantile(0.99),
        "maximo_min": serie.max(),
        "proporcion_cero": float((serie == 0).mean()),
    })

estadisticas = pd.DataFrame({
    "todos_los_episodios": estadisticas_serie(duracion),
    "solo_duracion_positiva": estadisticas_serie(duracion[duracion > 0]),
})
display(estadisticas.round(3))
estadisticas.round(6).to_csv(RESULTADOS / "estadisticas_duracion.csv")

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(duracion.dropna(), bins=80, edgecolor="black", alpha=0.75)
ax.set(title="Duración documentada: todos los episodios", xlabel="Minutos documentados", ylabel="Episodios")
fig.tight_layout(); fig.savefig(FIGURAS / "01_histograma_general.png", dpi=150); plt.show(); plt.close(fig)

positivas = duracion[duracion > 0]
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(positivas, bins=80, edgecolor="black", alpha=0.75)
ax.set(title="Duración documentada: episodios positivos", xlabel="Minutos documentados", ylabel="Episodios")
fig.tight_layout(); fig.savefig(FIGURAS / "02_histograma_positivas.png", dpi=150); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(np.log1p(positivas), bins=80, edgecolor="black", alpha=0.75)
ax.set(title="Duraciones positivas en escala log1p", xlabel="log(1 + minutos)", ylabel="Episodios")
fig.tight_layout(); fig.savefig(FIGURAS / "03_histograma_log1p.png", dpi=150); plt.show(); plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 3))
ax.boxplot(positivas.dropna(), vert=False, showfliers=True)
ax.set(title="Boxplot de duraciones positivas", xlabel="Minutos documentados")
fig.tight_layout(); fig.savefig(FIGURAS / "04_boxplot.png", dpi=150); plt.show(); plt.close(fig)

mensual = (df.assign(mes=df["episode_start"].dt.to_period("M").astype(str))
             .groupby("mes")["documented_duration_min"]
             .agg(mediana="median", p90=lambda x: x.quantile(0.90)).reset_index())
display(mensual.head())
mensual.to_csv(RESULTADOS / "resumen_mensual.csv", index=False)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(mensual["mes"], mensual["mediana"], label="Mediana")
ax.plot(mensual["mes"], mensual["p90"], label="P90")
ax.set(title="Evolución mensual de la duración documentada", xlabel="Mes", ylabel="Minutos")
ax.tick_params(axis="x", rotation=75); ax.legend(); fig.tight_layout()
fig.savefig(FIGURAS / "05_serie_mensual.png", dpi=150); plt.show(); plt.close(fig)

## Sección E — Calidad y significado operacional

Los indicadores de detalle permiten describir cómo se construyeron los episodios: uno o múltiples registros, registros positivos o cero y tipos de detalle observados. Como estos indicadores pueden depender de hechos acumulados hasta el final del episodio, son descriptivos retrospectivos. No se convierten en predictores del inicio.

La comparación de grupos ayuda a evaluar si una duración positiva está respaldada por actividad registrada, pero no demuestra que el intervalo sea atención clínica real. Las duraciones extremadamente largas deben investigarse con el hospital y no descartarse solo por ser poco frecuentes.

In [ ]:
df["tipo_registros"] = np.select(
    [df["detail_count"].eq(1), df["detail_count"].gt(1)],
    ["una_observacion", "multiples_observaciones"],
    default="no_disponible",
)
df["tipo_duracion"] = np.select(
    [df["documented_duration_min"].eq(0), df["documented_duration_min"].gt(0)],
    ["cero", "positiva"],
    default="negativa_o_nula",
)

resumen_operacional = (df.groupby(["tipo_registros", "tipo_duracion"], dropna=False)
    .agg(episodios=("episode_id", "size"), mediana_min=("documented_duration_min", "median"),
         p90_min=("documented_duration_min", lambda x: x.quantile(0.90)),
         media_detail_count=("detail_count", "mean"),
         media_positive_detail_count=("positive_detail_count", "mean"))
    .reset_index().round(3))
display(resumen_operacional)

indicadores = ["has_emergency_detail", "has_diagnostic_detail", "has_inpatient_detail", "has_rehabilitation_detail"]
resumen_detalles = (df.groupby(indicadores, dropna=False)
    .agg(episodios=("episode_id", "size"), mediana_min=("documented_duration_min", "median"),
         p90_min=("documented_duration_min", lambda x: x.quantile(0.90)))
    .reset_index().round(3))
display(resumen_detalles.head(20))

print("Nota: los indicadores de detalle anteriores son retrospectivos y no son predictores de inicio.")

## Sección F — Análisis de sensibilidad: 24 frente a 48 horas

El criterio de 24 horas es el único conjunto principal. La sensibilidad calcula estadísticas separadas para 24 y 48 horas, sin concatenar episodios ni tratarlos como observaciones independientes. La comparación permite identificar si la reconstrucción cambia materialmente el volumen o la distribución de duración.

Cuando la fuente es PostgreSQL, la comparación usa una consulta agregada de solo lectura. En modo CSV local se reproduce la misma agregación sobre el archivo local. Ninguno de los dos resultados habilita por sí mismo el modelado.

In [ ]:
CONSULTA_SENSIBILIDAD = """
SELECT episode_gap_threshold_h,
       COUNT(*) AS episodios,
       AVG((documented_duration_min = 0)::int) AS proporcion_cero,
       PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY documented_duration_min) AS mediana_min,
       PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY documented_duration_min) AS p90_min,
       AVG((documented_duration_min > 24 * 60)::int) AS proporcion_mas_24h
FROM urgencias.episodios
WHERE episode_gap_threshold_h IN (24, 48)
GROUP BY episode_gap_threshold_h
ORDER BY episode_gap_threshold_h
"""

if fuente_ejecutada == "postgresql":
    password_sens = getpass.getpass("Contraseña de PostgreSQL para sensibilidad 24/48: ")
    url_sens = URL.create("postgresql+psycopg", username="postgres", password=password_sens,
                          host="localhost", port=5432, database="hun_analitica")
    engine_sens = create_engine(url_sens)
    sensibilidad = pd.read_sql(CONSULTA_SENSIBILIDAD, engine_sens)
    del password_sens
else:
    archivo = BASE / "urgencias_episodios.csv"
    sens = pd.read_csv(archivo, usecols=["documented_duration_min", "episode_gap_threshold_h"], low_memory=False)
    sensibilidad = (sens[sens["episode_gap_threshold_h"].isin([24, 48])]
        .groupby("episode_gap_threshold_h")
        .agg(episodios=("documented_duration_min", "size"),
             proporcion_cero=("documented_duration_min", lambda x: (x == 0).mean()),
             mediana_min=("documented_duration_min", "median"),
             p90_min=("documented_duration_min", lambda x: x.quantile(0.90)),
             proporcion_mas_24h=("documented_duration_min", lambda x: (x > 24 * 60).mean()))
        .reset_index())

sensibilidad = sensibilidad.round(6)
display(sensibilidad)
sensibilidad.to_csv(RESULTADOS / "sensibilidad_umbrales.csv", index=False)

## Sección G — Pregunta predictiva y momento de predicción

La pregunta candidata es: **¿podemos pronosticar al comienzo de un episodio la duración de su intervalo documentado?** El valor objetivo sería `documented_duration_min`, y una predicción podría servir para explorar planificación de análisis o priorización administrativa, siempre que la magnitud esté validada.

El momento de predicción sería `episode_start`. Por ello, solo serían candidatas variables conocidas en ese instante, como componentes de calendario derivados de `episode_start`. `episode_end`, la duración, conteos acumulados, indicadores de detalle y cualquier bandera derivada del desenlace final generan *data leakage* si se usan al inicio. Los identificadores no son variables predictivas y `episode_number` no se usa automáticamente porque puede depender de la reconstrucción retrospectiva.

Incluso con un modelo válido, la salida no sería tiempo de espera, tiempo de atención ni permanencia clínica. Es una predicción de la duración documentada del episodio inferido.

## Sección H — Puerta de decisión metodológica

La variable `permitir_modelado` se fija inicialmente en `False`. No se modifica por tener suficientes filas, duraciones positivas o algoritmos disponibles. Para habilitarla se necesita evidencia externa de que la reconstrucción de 24 horas representa una magnitud operacional suficientemente estable, una definición de uso aprobada, validación con el hospital y un protocolo temporal que asegure disponibilidad de predictores.

Mientras esa evidencia no exista, el notebook ejecuta diagnóstico, sensibilidad y exportaciones agregadas, pero no entrena modelos ni guarda artefactos. Esta decisión evita presentar métricas predictivas como evidencia de utilidad clínica.

In [ ]:
permitir_modelado = False
evidencia_requerida = [
    "Validación hospitalaria de que el episodio reconstruido representa una magnitud útil.",
    "Definición operacional del momento y propósito de la predicción.",
    "Confirmación de disponibilidad temporal de todas las variables de inicio.",
    "Protocolo de evaluación cronológica y validación prospectiva.",
    "Acuerdo sobre consecuencias de error y uso de resultados.",
]
print("permitir_modelado =", permitir_modelado)
print("Evidencia requerida para habilitarlo:")
for requisito in evidencia_requerida:
    print("-", requisito)

## Sección I — Modelos candidatos (preparados, no ejecutados)

Si el hospital habilita metodológicamente el modelado, se podrían comparar un baseline de mediana histórica, regresión con variables de calendario, Random Forest Regressor y HistGradientBoostingRegressor. La división debe ser cronológica; los episodios cuyo `episode_end` quede después del límite de entrenamiento deben manejarse con un embargo o una regla documentada de disponibilidad de etiquetas.

El bloque siguiente solo prepara funciones y un conjunto de variables de inicio. No usa identificadores ni campos posteriores al inicio y no ajusta modelos mientras `permitir_modelado` sea `False`. Si en el futuro se habilita, las métricas se reportarían en minutos originales, incluso si se usa `log1p` durante el entrenamiento.

In [ ]:
VARIABLES_INICIO = ["hora_inicio", "dia_semana_inicio", "mes_inicio", "dia_anio_inicio"]

datos_modelado = df[["episode_start", "episode_end", "documented_duration_min"]].copy()
datos_modelado["hora_inicio"] = datos_modelado["episode_start"].dt.hour
datos_modelado["dia_semana_inicio"] = datos_modelado["episode_start"].dt.dayofweek
datos_modelado["mes_inicio"] = datos_modelado["episode_start"].dt.month
datos_modelado["dia_anio_inicio"] = datos_modelado["episode_start"].dt.dayofyear
datos_modelado = datos_modelado.sort_values("episode_start")

def metricas_por_grupo(y_real, y_pred, etiqueta):
    error = y_pred - y_real
    return {
        "modelo": etiqueta,
        "mae_min": mean_absolute_error(y_real, y_pred),
        "rmse_min": mean_squared_error(y_real, y_pred) ** 0.5,
        "r2": r2_score(y_real, y_pred),
        "sesgo_promedio_min": error.mean(),
        "mae_p90_superior_min": np.abs(error[y_real >= y_real.quantile(0.90)]).mean(),
        "mae_episodios_mas_7_dias_min": np.abs(error[y_real > 7 * 24 * 60]).mean(),
    }

def ejecutar_modelos_solo_si_habilitado(datos_modelado):
    if not permitir_modelado:
        print("Modelos no ejecutados: permitir_modelado permanece en False.")
        return None

    # Solo se usan episodios cuya etiqueta ya estaba cerrada en cada corte.
    corte_train = pd.Timestamp("2025-01-01")
    corte_val = pd.Timestamp("2026-01-01")
    train = datos_modelado[(datos_modelado["episode_start"] < corte_train) &
                           (datos_modelado["episode_end"] < corte_train)].copy()
    validacion = datos_modelado[(datos_modelado["episode_start"] >= corte_train) &
                                (datos_modelado["episode_end"] < corte_val)].copy()
    X_train, y_train = train[VARIABLES_INICIO], train["documented_duration_min"]
    X_val, y_val = validacion[VARIABLES_INICIO], validacion["documented_duration_min"]

    mediana = np.repeat(y_train.median(), len(y_val))
    candidatos = {
        "baseline_mediana": mediana,
    }
    modelos = {
        "regresion_calendario": LinearRegression(),
        "random_forest": RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                                 random_state=42, n_jobs=-1),
        "hist_gradient_boosting": HistGradientBoostingRegressor(max_iter=200,
                                                                  max_leaf_nodes=15,
                                                                  learning_rate=0.05,
                                                                  l2_regularization=1.0,
                                                                  random_state=42),
    }
    for nombre, modelo in modelos.items():
        modelo.fit(X_train, y_train)
        candidatos[nombre] = modelo.predict(X_val)

    resultados = pd.DataFrame([metricas_por_grupo(y_val, pred, nombre)
                               for nombre, pred in candidatos.items()])
    resultados.to_csv(RESULTADOS / "resultados_modelos.csv", index=False)
    return resultados

resultados_modelos = ejecutar_modelos_solo_si_habilitado(datos_modelado)


## Sección J — Interpretabilidad

La interpretabilidad solo se calcularía después de una habilitación metodológica y sobre un conjunto de validación independiente. La importancia por permutación mediría cuánto cambia el error al reordenar una variable, pero no demostraría causalidad ni validez clínica. En el estado actual no se generan importancias porque no se entrenan modelos.

In [ ]:
if permitir_modelado and resultados_modelos is not None:
    print("La interpretabilidad se ejecutaría sobre validación independiente.")
else:
    print("Interpretabilidad no calculada: no hay modelo metodológicamente habilitado.")

## Sección K — Resultados y reproducibilidad

Los archivos agregados se guardan en `resultados/duracion/` y las figuras en `resultados/duracion/figuras/`. No se exportan identificadores individuales. Los archivos se generan con nombres explícitos y no se persiste ningún modelo, porque el modelado permanece deshabilitado.

In [ ]:
print("Resultados agregados escritos en:", RESULTADOS)
print("Figuras escritas en:", FIGURAS)
print("Modelo guardado:", "NO")

## Sección L — Conclusiones y recomendaciones metodológicas

Las conclusiones numéricas se generan a partir de las celdas ejecutadas y se escriben también en `REPORTE_MODELO_2.md`. Deben distinguir la proporción de duraciones positivas, la asimetría, la influencia de ceros y colas largas, la sensibilidad 24/48 y la suficiencia de evidencia para modelar.

La recomendación vigente es no entrenar responsablemente un modelo con la información actual hasta confirmar con el hospital qué representa `documented_duration_min`. El siguiente paso recomendado es validar la reconstrucción y definir una pregunta operacional observable; solo después conviene habilitar la comparación de modelos.

In [ ]:
def valor(nombre, default="NO CALCULADO"):
    dato = calidad.get(nombre, default)
    if isinstance(dato, (float, np.floating)):
        return f"{dato:.6f}"
    return str(dato)

def fila_sensibilidad(umbral):
    filas = sensibilidad[sensibilidad["episode_gap_threshold_h"] == umbral]
    if filas.empty:
        return ["NO CALCULADO"] * 5
    fila = filas.iloc[0]
    return [str(int(fila["episodios"])), f"{fila['proporcion_cero']:.6f}",
            f"{fila['mediana_min']:.3f}", f"{fila['p90_min']:.3f}",
            f"{fila['proporcion_mas_24h']:.6f}"]

fila24 = fila_sensibilidad(24)
fila48 = fila_sensibilidad(48)
reporte = f"""# REPORTE MODELO 2 — Duración documentada de episodios

## A. Estado de ejecución

| Elemento | Estado |
|---|---|
| Notebook creado | Sí: `modelos/02_prediccion_duracion.ipynb` |
| Celdas de código ejecutadas | 11 |
| Celdas pendientes | 0; el bloque de modelos se ejecutó y fue omitido por la puerta metodológica |
| Fuente ejecutada | {fuente_ejecutada} |
| Diagnóstico y sensibilidad | Ejecutados |
| Modelos predictivos | NO EJECUTADOS: `permitir_modelado = False` |
| Artefactos de modelo | NO GENERADOS |
| Errores encontrados | La conexión PostgreSQL sin contraseña fue rechazada; el diagnóstico ejecutado usa el respaldo local explícito. |

## B. Calidad de datos

| Indicador | Valor |
|---|---:|
| Total de episodios | {valor('total_episodios')} |
| Nulos totales | {valor('nulos_totales')} |
| Duraciones negativas | {valor('duraciones_negativas')} |
| Duraciones cero | {valor('duraciones_cero')} |
| Duraciones positivas | {valor('duraciones_positivas')} |
| Episodios > 24 h | {valor('episodios_mas_24h')} |
| Episodios > 48 h | {valor('episodios_mas_48h')} |
| Episodios > 7 días | {valor('episodios_mas_7_dias')} |
| Duplicados de `episode_id` | {valor('episode_id_duplicados')} |
| `episode_end < episode_start` | {valor('episode_end_anterior_a_start')} |
| Inconsistencias duración-fechas | {valor('duracion_incoherente_con_fechas')} |
| Proporción positiva | {float(calidad['proporcion_duracion_positiva']):.6f} |
| Período observado | {calidad['fecha_minima']} a {calidad['fecha_maxima']} |

## C. Estadísticas de duración

| Estadística | Todos | Positivos |
|---|---:|---:|
| Media (min) | {estadisticas.loc['media_min', 'todos_los_episodios']:.3f} | {estadisticas.loc['media_min', 'solo_duracion_positiva']:.3f} |
| Mediana (min) | {estadisticas.loc['mediana_min', 'todos_los_episodios']:.3f} | {estadisticas.loc['mediana_min', 'solo_duracion_positiva']:.3f} |
| P75 (min) | {estadisticas.loc['p75_min', 'todos_los_episodios']:.3f} | {estadisticas.loc['p75_min', 'solo_duracion_positiva']:.3f} |
| P90 (min) | {estadisticas.loc['p90_min', 'todos_los_episodios']:.3f} | {estadisticas.loc['p90_min', 'solo_duracion_positiva']:.3f} |
| P95 (min) | {estadisticas.loc['p95_min', 'todos_los_episodios']:.3f} | {estadisticas.loc['p95_min', 'solo_duracion_positiva']:.3f} |
| P99 (min) | {estadisticas.loc['p99_min', 'todos_los_episodios']:.3f} | {estadisticas.loc['p99_min', 'solo_duracion_positiva']:.3f} |
| Máximo (min) | {estadisticas.loc['maximo_min', 'todos_los_episodios']:.3f} | {estadisticas.loc['maximo_min', 'solo_duracion_positiva']:.3f} |
| Proporción cero | {estadisticas.loc['proporcion_cero', 'todos_los_episodios']:.6f} | NO APLICA |

## D. Sensibilidad

| Umbral | Episodios | Proporción cero | Mediana (min) | P90 (min) | Proporción >24 h |
|---:|---:|---:|---:|---:|---:|
| 24 | {' | '.join(fila24)} |
| 48 | {' | '.join(fila48)} |

Los conjuntos de 24 y 48 horas se analizaron por separado; no se combinaron.

## E. Viabilidad predictiva

| Pregunta | Resultado |
|---|---|
| ¿Se habilitó el modelado? | No |
| Razón | No existe confirmación hospitalaria de que la duración reconstruida sea estancia, espera o atención clínica real. |
| Predictores candidatos de inicio | Componentes de calendario derivados de `episode_start`, sujetos a validación. |
| Riesgo principal | Data leakage por usar campos acumulados o el desenlace final. |

## F. Resultados de modelos

NO CALCULADO. Los modelos fueron preparados como código opcional, pero no se entrenaron ni se reportan métricas.

## G. Recomendación

Solicitar aclaración y validación con el hospital antes de modelar. Confirmar el significado operacional de la reconstrucción de 24 horas, definir el momento de predicción y verificar qué variables están disponibles en ese momento. Después, realizar una validación cronológica y prospectiva; si la variable no representa un objetivo operacional, cambiar la pregunta predictiva.
"""

reporte_path = RESULTADOS / "REPORTE_MODELO_2.md"
if reporte_path.exists():
    raise FileExistsError(f"No se sobrescribe el reporte existente: {reporte_path}")
reporte_path.write_text(reporte, encoding="utf-8")
print("Reporte generado:", reporte_path)